In [0]:
from pyspark.sql.functions import col, count, countDistinct, max
from datetime import datetime

# Читаем gold таблицы
daily_metrics_df = spark.table("weather_project.gold.daily_city_metrics")
features_df = spark.table("weather_project.gold.ml_features")
hourly_df = spark.table("weather_project.silver.hourly_clean")
daily_df = spark.table("weather_project.silver.daily_clean")

# Считаем метрики
metrics = daily_metrics_df.agg(
    count("*").alias("processed_rows"),
    countDistinct("city_name").alias("cities_count"),
    max("event_date").alias("last_processing_date")
).collect()[0]

processed_rows = metrics["processed_rows"]
cities_count = metrics["cities_count"]
last_processing_date = metrics["last_processing_date"]

print(f"Processed rows: {processed_rows}")
print(f"Cities count: {cities_count}")
print(f"Last processing date: {last_processing_date}")

# Записываем в monitoring
report = [(
    processed_rows,
    cities_count,
    str(last_processing_date),
    datetime.now()
)]

schema = ["processed_rows", "cities_count", "last_processing_date", "reported_at"]
report_df = spark.createDataFrame(report, schema)

report_df.write \
    .mode("append") \
    .saveAsTable("weather_project.monitoring.pipeline_reports")

print(f"\n Report written to monitoring.pipeline_reports")
display(report_df)